# Import Important Libraries

In [7]:
# PROJECT: Student Subscription & Churn Analysis
# ROLE: Data Analyst
# Import all required libraries
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

print("Libraries loaded!")

Libraries loaded!


In [8]:
# SETUP: Create project folders

# Get current directory
current_dir = os.getcwd()

# Define paths
data_dir = os.path.abspath(
    os.path.join(current_dir, '..', 'data')
)

# Create folders if they don't exist
os.makedirs(data_dir, exist_ok=True)

print(f"Data folder ready: {data_dir}")

Data folder ready: G:\01_Learning\Projects\01_My Projects\08_Chegg Inc\data


# Generate Student Data

In [10]:
# FILE: subscriptions.csv
# PURPOSE: Every subscription record per student
# REAL USE: Created when student purchased plan
#           Updated when student cancelled
# ROWS: ~12,000-15,000 subscriptions
#
# KEY BUSINESS LOGIC:
# Post May 2023 (AI disruption) churn increased
# Annual plan churns least
# Monthly plan churns most

PLAN_CONFIG = {
    'Monthly': {
        'price':    9.99,
        'duration': 30,
        'weight':   0.55,
    },
    'Quarterly': {
        'price':    19.99,
        'duration': 90,
        'weight':   0.25,
    },
    'Annual': {
        'price':    59.99,
        'duration': 365,
        'weight':   0.20,
    },
}

CHURN_REASONS = {
    'Found Better Alternative': 0.30,
    'Price Too High':           0.25,
    'Not Useful Anymore':       0.20,
    'Exam Season Over':         0.15,
    'Technical Issues':         0.05,
    'Other':                    0.05,
}

print("Generating subscriptions.csv...")

rows   = []
sub_id = 1

for _, student in students_df.iterrows():

    reg_date = pd.Timestamp(
        student['Registration_Date']
    )
    days_to_sub = random.randint(1, 30)
    first_sub_date = reg_date + timedelta(
        days=days_to_sub
    )

    # 10% students never subscribed
    if random.random() < 0.10:
        continue

    plan   = random.choices(
        list(PLAN_CONFIG.keys()),
        weights=[v['weight']
                 for v in PLAN_CONFIG.values()]
    )[0]
    config   = PLAN_CONFIG[plan]
    duration = config['duration']

    if plan == 'Annual':
        max_subs = random.choices(
            [1, 2], weights=[80, 20])[0]
    elif plan == 'Quarterly':
        max_subs = random.choices(
            [1, 2, 3], weights=[50, 30, 20])[0]
    else:
        max_subs = random.choices(
            [1, 2, 3, 4, 5],
            weights=[35, 25, 20, 12, 8]
        )[0]

    current_start = first_sub_date

    for sub_num in range(max_subs):

        sub_start = current_start
        sub_end   = sub_start + timedelta(
            days=duration
        )

        if sub_start > datetime(2024, 6, 30):
            break

        # Churn probability increases post AI
        if sub_start < datetime(2023, 5, 1):
            churn_prob = 0.20
        elif sub_start < datetime(2023, 11, 1):
            churn_prob = 0.45
        else:
            churn_prob = 0.65

        if plan == 'Annual':
            churn_prob *= 0.5
        elif plan == 'Quarterly':
            churn_prob *= 0.75

        if sub_num == max_subs - 1:
            churn_prob = min(
                churn_prob * 1.3, 0.95
            )

        churned = random.random() < churn_prob

        if churned:
            churn_after  = random.randint(
                3, duration - 1
            )
            churn_date   = sub_start + timedelta(
                days=churn_after
            )
            actual_end   = churn_date
            days_active  = churn_after
            churn_reason = random.choices(
                list(CHURN_REASONS.keys()),
                weights=list(CHURN_REASONS.values())
            )[0]
            # Post Nov 2023 — AI alternative more common
            if sub_start > datetime(2023, 11, 1):
                if random.random() < 0.40:
                    churn_reason = \
                        'Found Better Alternative'
        else:
            actual_end   = sub_end
            churn_date   = None
            days_active  = duration
            churn_reason = None

        rows.append({
            'Sub_ID':          f'SUB_{sub_id:05d}',
            'Student_ID':      student['Student_ID'],
            'Country':         student['Country'],
            'Age_Group':       student['Age_Group'],
            'Education_Level': student['Education_Level'],
            'Plan_Type':       plan,
            'Plan_Price_USD':  config['price'],
            'Sub_Start_Date':  sub_start.date(),
            'Sub_End_Date':    actual_end.date(),
            'Sub_Start_Month': sub_start.strftime(
                '%Y-%m'),
            'Sub_Start_Year':  sub_start.year,
            'Churned':         'Yes' if churned
                               else 'No',
            'Churn_Reason':    churn_reason,
            'Churn_Date':      churn_date.date()
                               if churn_date
                               else None,
            'Days_Active':     days_active,
            'Renewal_Number':  sub_num + 1,
            'Revenue_USD':     round(
                config['price'] *
                (days_active / duration), 2
            ),
        })
        sub_id += 1

        if not churned:
            current_start = sub_end
        else:
            break

subs_df = pd.DataFrame(rows)

# Verification
print(f"\n Total Subscriptions: {len(subs_df):,}")
print(f"   Unique Students: "
      f"{subs_df['Student_ID'].nunique():,}")

print(f"\n Plan Distribution:")
print(subs_df['Plan_Type'].value_counts())

print(f"\n Churned vs Retained:")
print(subs_df['Churned'].value_counts())
overall = (subs_df['Churned']=='Yes').mean()*100
print(f"   Overall Churn Rate: {overall:.1f}%")

print(f"\n Churn Rate by Plan:")
plan_churn = subs_df.groupby(
    'Plan_Type', observed=True
).apply(
    lambda x: round(
        (x['Churned']=='Yes').sum()
        / len(x) * 100, 1
    )
).reset_index()
plan_churn.columns = ['Plan_Type', 'Churn_Rate']
print(plan_churn.to_string(index=False))

print(f"\n Churn Reasons (top 3):")
print(subs_df[subs_df['Churned']=='Yes']
      ['Churn_Reason'].value_counts().head(3))

print(f"\n Revenue Stats:")
print(f"   Total: ${subs_df['Revenue_USD'].sum():,.2f}")
print(f"   Avg:   ${subs_df['Revenue_USD'].mean():.2f}")

# Save
save_path = os.path.join(
    data_dir, 'subscriptions.csv')
subs_df.to_csv(save_path, index=False)
print(f"\n subscriptions.csv saved! "
      f"Rows: {len(subs_df):,}")

Generating subscriptions.csv...

 Total Subscriptions: 12,761
   Unique Students: 8,856

 Plan Distribution:
Plan_Type
Monthly      7762
Quarterly    3099
Annual       1900
Name: count, dtype: int64

 Churned vs Retained:
Churned
No     7885
Yes    4876
Name: count, dtype: int64
   Overall Churn Rate: 38.2%

 Churn Rate by Plan:
Plan_Type  Churn_Rate
   Annual        26.6
  Monthly        42.3
Quarterly        35.2

 Churn Reasons (top 3):
Churn_Reason
Found Better Alternative    2159
Price Too High               969
Not Useful Anymore           761
Name: count, dtype: int64

 Revenue Stats:
   Total: $212,284.47
   Avg:   $16.64

 subscriptions.csv saved! Rows: 12,761


C:\Users\Grish\AppData\Local\Temp\ipykernel_9468\2958343973.py:185: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


# Generate Subscription Data

In [11]:
# FILE: subscriptions.csv
# PURPOSE: Every subscription record per student
# REAL USE: Created when student purchased plan
#           Updated when student cancelled
# ROWS: ~12,000-15,000 subscriptions
#
# KEY BUSINESS LOGIC:
# Post May 2023 (AI disruption) churn increased
# Annual plan churns least
# Monthly plan churns most


PLAN_CONFIG = {
    'Monthly': {
        'price':    9.99,
        'duration': 30,
        'weight':   0.55,   # 55% students monthly choose karte
    },
    'Quarterly': {
        'price':    19.99,
        'duration': 90,
        'weight':   0.25,
    },
    'Annual': {
        'price':    59.99,
        'duration': 365,
        'weight':   0.20,
    },
}

CHURN_REASONS = {
    'Price Too High':        0.30,
    'Found Better Alternative': 0.25,  # ChatGPT!
    'Not Useful Anymore':    0.20,
    'Exam Season Over':      0.15,
    'Technical Issues':      0.05,
    'Other':                 0.05,
}

print("subscriptions.csv generating...")

# Generate subscriptions
rows   = []
sub_id = 1

for _, student in students_df.iterrows():

    reg_date = pd.Timestamp(student['Registration_Date'])

    # Not every student subscribes immediately
    # Some take 1-30 days after registration
    days_to_sub = random.randint(1, 30)
    first_sub_date = reg_date + timedelta(days=days_to_sub)

    # Some students never subscribed (10%)
    if random.random() < 0.10:
        continue

    # Choose plan
    plan = random.choices(
        list(PLAN_CONFIG.keys()),
        weights=[v['weight'] for v in PLAN_CONFIG.values()]
    )[0]
    config   = PLAN_CONFIG[plan]
    duration = config['duration']

    # How many times did student renew?
    # Annual → mostly 1 subscription
    # Monthly → can have multiple renewals
    if plan == 'Annual':
        max_subs = random.choices([1, 2], weights=[80, 20])[0]
    elif plan == 'Quarterly':
        max_subs = random.choices([1, 2, 3], weights=[50, 30, 20])[0]
    else:
        max_subs = random.choices(
            [1, 2, 3, 4, 5],
            weights=[35, 25, 20, 12, 8]
        )[0]

    current_start = first_sub_date

    for sub_num in range(max_subs):

        sub_start = current_start
        sub_end   = sub_start + timedelta(days=duration)

        # Cap at Jun 2024
        if sub_start > datetime(2024, 6, 30):
            break

        # Churn Logic
        # Post ChatGPT (May 2023) churn increased
        # Post Google AI (Nov 2023) churn even more
        if sub_start < datetime(2023, 5, 1):
            churn_prob = 0.20   # Pre-AI era
        elif sub_start < datetime(2023, 11, 1):
            churn_prob = 0.45   # ChatGPT impact
        else:
            churn_prob = 0.65   # Google AI impact

        # Annual subscribers churn less
        if plan == 'Annual':
            churn_prob *= 0.5
        elif plan == 'Quarterly':
            churn_prob *= 0.75

        # Last subscription more likely to churn
        if sub_num == max_subs - 1:
            churn_prob = min(churn_prob * 1.3, 0.95)

        churned = random.random() < churn_prob

        if churned:
            # Churn happens somewhere in middle
            churn_after = random.randint(
                3, duration - 1
            )
            churn_date  = sub_start + timedelta(
                days=churn_after
            )
            actual_end  = churn_date
            days_active = churn_after

            churn_reason = random.choices(
                list(CHURN_REASONS.keys()),
                weights=list(CHURN_REASONS.values())
            )[0]

            # Post Nov 2023 — "Found Better Alternative"
            # more common (ChatGPT/Gemini)
            if sub_start > datetime(2023, 11, 1):
                if random.random() < 0.40:
                    churn_reason = 'Found Better Alternative'

        else:
            actual_end   = sub_end
            churn_date   = None
            days_active  = duration
            churn_reason = None

        rows.append({
            'Sub_ID':          f'SUB_{sub_id:05d}',
            'Student_ID':      student['Student_ID'],
            'Country':         student['Country'],
            'Age_Group':       student['Age_Group'],
            'Education_Level': student['Education_Level'],
            'Plan_Type':       plan,
            'Plan_Price_USD':  config['price'],
            'Sub_Start_Date':  sub_start.date(),
            'Sub_End_Date':    actual_end.date(),
            'Sub_Start_Month': sub_start.strftime('%Y-%m'),
            'Sub_Start_Year':  sub_start.year,
            'Churned':         'Yes' if churned else 'No',
            'Churn_Reason':    churn_reason,
            'Churn_Date':      churn_date.date()
                               if churn_date else None,
            'Days_Active':     days_active,
            'Renewal_Number':  sub_num + 1,
            'Revenue_USD':     round(
                config['price'] * (days_active / duration),
                2
            ),
        })

        sub_id += 1

        # Next subscription starts after this ends
        if not churned:
            current_start = sub_end
        else:
            break   # Churned — no more renewals

# Create DataFrame
subs_df = pd.DataFrame(rows)

# Verification
print(f"\n Total Subscriptions: {len(subs_df):,}")
print(f"   Unique Students:    "
      f"{subs_df['Student_ID'].nunique():,}")

print(f"\n Plan Distribution:")
print(subs_df['Plan_Type'].value_counts())

print(f"\n Churned vs Retained:")
print(subs_df['Churned'].value_counts())

overall_churn = (
    subs_df['Churned']=='Yes'
).sum() / len(subs_df) * 100
print(f"   Overall Churn Rate: {overall_churn:.1f}%")

print(f"\n Churn Reasons (churned only):")
churned = subs_df[subs_df['Churned']=='Yes']
print(churned['Churn_Reason'].value_counts())

print(f"\n Churn Rate by Plan:")
plan_churn = subs_df.groupby('Plan_Type').apply(
    lambda x: (x['Churned']=='Yes').sum() / len(x) * 100
).round(1)
print(plan_churn)

print(f"\n Revenue Stats:")
print(f"   Total Revenue: "
      f"${subs_df['Revenue_USD'].sum():,.2f}")
print(f"   Avg per Sub:   "
      f"${subs_df['Revenue_USD'].mean():.2f}")

print(f"\n First 5 rows:")
print(subs_df[[
    'Sub_ID', 'Student_ID', 'Plan_Type',
    'Sub_Start_Date', 'Churned',
    'Churn_Reason', 'Days_Active'
]].head().to_string(index=False))

# Save
save_path = os.path.join(data_dir, 'subscriptions.csv')
subs_df.to_csv(save_path, index=False)

print(f"\n subscriptions.csv saved! "
      f"Rows: {len(subs_df):,}")

subscriptions.csv generating...

 Total Subscriptions: 12,626
   Unique Students:    8,842

 Plan Distribution:
Plan_Type
Monthly      7615
Quarterly    3111
Annual       1900
Name: count, dtype: int64

 Churned vs Retained:
Churned
No     7727
Yes    4899
Name: count, dtype: int64
   Overall Churn Rate: 38.8%

 Churn Reasons (churned only):
Churn_Reason
Found Better Alternative    1910
Price Too High              1195
Not Useful Anymore           785
Exam Season Over             617
Other                        206
Technical Issues             186
Name: count, dtype: int64

 Churn Rate by Plan:
Plan_Type
Annual       27.1
Monthly      42.8
Quarterly    36.3
dtype: float64

 Revenue Stats:
   Total Revenue: $210,929.21
   Avg per Sub:   $16.71

 First 5 rows:
   Sub_ID Student_ID Plan_Type Sub_Start_Date Churned             Churn_Reason  Days_Active
SUB_00001  STU_00001    Annual     2024-03-27     Yes Found Better Alternative           92
SUB_00002  STU_00004   Monthly     2023-09-08 

C:\Users\Grish\AppData\Local\Temp\ipykernel_9468\822162572.py:197: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  plan_churn = subs_df.groupby('Plan_Type').apply(


# Generate MAU Monthly Data

In [12]:
# FILE: mau_monthly.csv
# PURPOSE: Monthly business KPIs
# REAL USE: This was the main dashboard table
#             CFO and leadership watched this
#             every single month
# ROWS: 18 months (Jan 2023 to Jun 2024)
#
# KEY INSIGHT:
# Search traffic dropped FIRST (May 2023)
# Subscriptions dropped 2-3 months LATER
# This is what team tracked live


# Base numbers (real Edtech scale)
BASE_MAU        = 4_500_000   # Jan 2023 peak
BASE_REVENUE    = 18_000_000  # monthly revenue USD
BASE_CSAT       = 4.2         # out of 5

print("Generating mau_monthly.csv...")

rows = []

for year in [2023, 2024]:
    for month in range(1, 13):

        # Stop at Jun 2024
        if year == 2024 and month > 6:
            break

        # Months since Jan 2023
        m = (year - 2023) * 12 + month

        # MAU decline logic
        # Phase 1: Jan-Apr 2023 — Stable
        # Phase 2: May-Aug 2023 — Slow decline
        # Phase 3: Sep-Dec 2023 — Steep decline
        # Phase 4: 2024 — Crisis

        if m <= 4:
            # Stable phase
            mau_factor = 1.0 + random.uniform(
                -0.02, 0.02)
            traffic_factor = 1.0 + random.uniform(
                -0.02, 0.02)
            csat_drop = 0.0

        elif m <= 8:
            # ChatGPT impact begins
            mau_factor = (
                1.0 - (m - 4) * 0.04
                + random.uniform(-0.01, 0.01)
            )
            # Traffic drops faster than MAU
            traffic_factor = (
                1.0 - (m - 4) * 0.07
                + random.uniform(-0.02, 0.02)
            )
            csat_drop = (m - 4) * 0.05

        elif m <= 12:
            # Steep decline
            mau_factor = (
                0.84 - (m - 8) * 0.07
                + random.uniform(-0.02, 0.02)
            )
            traffic_factor = (
                0.72 - (m - 8) * 0.08
                + random.uniform(-0.02, 0.02)
            )
            csat_drop = 0.20 + (m - 8) * 0.08

        else:
            # Crisis phase 2024
            mau_factor = max(
                0.30,
                0.56 - (m - 12) * 0.05
                + random.uniform(-0.02, 0.02)
            )
            traffic_factor = max(
                0.25,
                0.40 - (m - 12) * 0.04
                + random.uniform(-0.02, 0.02)
            )
            csat_drop = 0.52 + (m - 12) * 0.06

        # Calculate KPIs
        mau = int(BASE_MAU * mau_factor)

        # New subscribers — declining over time
        new_subs = int(
            mau * random.uniform(0.04, 0.07)
            * mau_factor
        )

        # Churned count — increasing over time
        churn_rate = min(
            0.15,
            0.05 + (1 - mau_factor) * 0.15
        )
        churned_count = int(mau * churn_rate)

        # Revenue follows MAU
        revenue = int(
            BASE_REVENUE * mau_factor
            * random.uniform(0.95, 1.05)
        )

        # CSAT declining
        csat = round(
            max(2.5,
                BASE_CSAT
                - csat_drop
                + random.uniform(-0.1, 0.1)),
            2
        )

        # Search traffic index
        # (100 = Jan 2023 baseline)
        traffic_index = round(
            max(25, 100 * traffic_factor), 1
        )

        rows.append({
            'Year':                year,
            'Month':               month,
            'Month_Label':         f'{year}-{month:02d}',
            'MAU':                 mau,
            'Search_Traffic_Index':traffic_index,
            'New_Subscribers':     new_subs,
            'Churned_Count':       churned_count,
            'Net_Change':          new_subs - churned_count,
            'Revenue_USD':         revenue,
            'Avg_CSAT':            csat,
            'Churn_Rate_Pct':      round(
                churned_count / mau * 100, 2),
        })

# Create DataFrame
mau_df = pd.DataFrame(rows)

# Verification
print(f"\n Total months: {len(mau_df)}")

print(f"\n MAU Trend:")
print(mau_df[['Month_Label', 'MAU',
              'Search_Traffic_Index',
              'Avg_CSAT']].to_string(index=False))

print(f"\n Revenue Trend:")
print(mau_df[['Month_Label',
              'Revenue_USD',
              'New_Subscribers',
              'Churned_Count',
              'Churn_Rate_Pct'
              ]].to_string(index=False))

print(f"\n Key Stats:")
print(f"   Peak MAU:    "
      f"{mau_df['MAU'].max():,} "
      f"({mau_df.loc[mau_df['MAU'].idxmax(),'Month_Label']})")
print(f"   Lowest MAU:  "
      f"{mau_df['MAU'].min():,} "
      f"({mau_df.loc[mau_df['MAU'].idxmin(),'Month_Label']})")
print(f"   MAU Drop:    "
      f"{(1 - mau_df['MAU'].min()/mau_df['MAU'].max())*100:.1f}%")
print(f"   Peak CSAT:   {mau_df['Avg_CSAT'].max()}")
print(f"   Lowest CSAT: {mau_df['Avg_CSAT'].min()}")
print(f"   Total Revenue: "
      f"${mau_df['Revenue_USD'].sum():,}")

# Save
save_path = os.path.join(data_dir, 'mau_monthly.csv')
mau_df.to_csv(save_path, index=False)

print(f"\n mau_monthly.csv saved! "
      f"Rows: {len(mau_df)}")

Generating mau_monthly.csv...

 Total months: 18

 MAU Trend:
Month_Label     MAU  Search_Traffic_Index  Avg_CSAT
    2023-01 4428383                 101.3      4.17
    2023-02 4522289                  98.9      4.16
    2023-03 4471018                 101.5      4.26
    2023-04 4541086                  98.5      4.27
    2023-05 4283894                  91.0      4.12
    2023-06 4124636                  87.0      4.18
    2023-07 4002689                  77.4      3.97
    2023-08 3820605                  73.8      4.03
    2023-09 3545791                  65.0      3.86
    2023-10 3230414                  54.0      3.92
    2023-11 2812771                  48.5      3.66
    2023-12 2604165                  39.9      3.66
    2024-01 2373543                  37.6      3.65
    2024-02 2085147                  33.1      3.46
    2024-03 1779658                  27.5      3.50
    2024-04 1568597                  25.0      3.36
    2024-05 1350000                  25.0      3.34
  

# Generate Student Activity

In [13]:
# FILE: student_activity.csv
# PURPOSE: Monthly activity per active student
# REAL EDtech: Tracked every login, question view,
#             and expert session automatically
# ROWS: ~80,000 (students x active months)
#
# BUSINESS USE:
# Low activity students → at risk of churn
# High activity → likely to renew


# All months we track
ALL_MONTHS = []
for year in [2023, 2024]:
    for month in range(1, 13):
        if year == 2024 and month > 6:
            break
        ALL_MONTHS.append(f'{year}-{month:02d}')

print(f" Tracking months: {len(ALL_MONTHS)}")
print(f"   From: {ALL_MONTHS[0]} To: {ALL_MONTHS[-1]}")

# Exam months — activity spikes
# Students use Chegg more during exam season
EXAM_MONTHS = [1, 4, 5, 11, 12]  # Jan,Apr,May,Nov,Dec

print("\n student_activity.csv generating...")

# Generate activity rows
rows = []

# Only process subscribed students
subscribed_students = subs_df['Student_ID'].unique()

for student_id in subscribed_students:

    # Get this student's subscriptions
    student_subs = subs_df[
        subs_df['Student_ID'] == student_id
    ].copy()

    # Get student profile
    student_info = students_df[
        students_df['Student_ID'] == student_id
    ]
    if len(student_info) == 0:
        continue

    student_info = student_info.iloc[0]
    age_group    = student_info['Age_Group']
    education    = student_info['Education_Level']
    country      = student_info['Country']

    # Base activity level — varies by student type
    # PhD students use it more than undergrads
    if education == 'PhD':
        base_logins   = random.randint(15, 25)
        base_q_viewed = random.randint(20, 40)
        base_sessions = random.randint(5, 12)
    elif education == 'Postgraduate':
        base_logins   = random.randint(10, 20)
        base_q_viewed = random.randint(12, 28)
        base_sessions = random.randint(3, 8)
    else:
        base_logins   = random.randint(6, 15)
        base_q_viewed = random.randint(8, 20)
        base_sessions = random.randint(1, 5)

    for month_label in ALL_MONTHS:

        year  = int(month_label[:4])
        month = int(month_label[5:])

        # Check if student had active subscription
        # during this month
        month_start = datetime(year, month, 1)
        if month == 12:
            month_end = datetime(year+1, 1, 1)
        else:
            month_end = datetime(year, month+1, 1)

        # Was student subscribed this month?
        was_subscribed = False
        for _, sub in student_subs.iterrows():
            sub_start = pd.Timestamp(sub['Sub_Start_Date'])
            sub_end   = pd.Timestamp(sub['Sub_End_Date'])
            if (sub_start < month_end and
                    sub_end > month_start):
                was_subscribed = True
                break

        if not was_subscribed:
            continue   # No activity if not subscribed

        # Activity multipliers
        # Exam months → more activity
        exam_boost = 1.5 if month in EXAM_MONTHS else 1.0

        # Summer months → less activity
        summer_drop = 0.6 if month in [6, 7, 8] else 1.0

        # Post ChatGPT → declining engagement
        if month_label >= '2023-05':
            months_since_chatgpt = (
                (year - 2023) * 12 + month - 5
            )
            ai_drop = max(
                0.4,
                1.0 - months_since_chatgpt * 0.04
            )
        else:
            ai_drop = 1.0

        # Combined multiplier
        multiplier = exam_boost * summer_drop * ai_drop

        # Calculate activity metrics
        logins = max(0, int(
            base_logins * multiplier
            + random.gauss(0, 2)
        ))
        q_viewed = max(0, int(
            base_q_viewed * multiplier
            + random.gauss(0, 3)
        ))
        expert_sessions = max(0, int(
            base_sessions * multiplier
            + random.gauss(0, 1)
        ))

        # Is student active this month?
        # Active = at least 1 login
        is_active = 1 if logins > 0 else 0

        # Activity score — 0 to 100
        # Used to predict churn risk
        activity_score = min(100, round(
            (logins * 2 +
             q_viewed * 1.5 +
             expert_sessions * 5) / 3
        , 1))

        # Churn risk based on activity
        if activity_score < 20:
            churn_risk = 'High'
        elif activity_score < 50:
            churn_risk = 'Medium'
        else:
            churn_risk = 'Low'

        rows.append({
            'Student_ID':      student_id,
            'Country':         country,
            'Age_Group':       age_group,
            'Education_Level': education,
            'Month_Label':     month_label,
            'Year':            year,
            'Month':           month,
            'Login_Count':     logins,
            'Questions_Viewed':q_viewed,
            'Expert_Sessions': expert_sessions,
            'Is_Active':       is_active,
            'Activity_Score':  activity_score,
            'Churn_Risk':      churn_risk,
        })

# Create DataFrame
activity_df = pd.DataFrame(rows)

# Verification
print(f"\n Total activity rows: {len(activity_df):,}")
print(f"   Unique students:    "
      f"{activity_df['Student_ID'].nunique():,}")
print(f"   Months tracked:     "
      f"{activity_df['Month_Label'].nunique()}")

print(f"\n Activity Stats:")
print(f"   Avg Logins/Month:   "
      f"{activity_df['Login_Count'].mean():.1f}")
print(f"   Avg Q Viewed/Month: "
      f"{activity_df['Questions_Viewed'].mean():.1f}")
print(f"   Avg Sessions/Month: "
      f"{activity_df['Expert_Sessions'].mean():.1f}")
print(f"   Avg Activity Score: "
      f"{activity_df['Activity_Score'].mean():.1f}")

print(f"\n Churn Risk Distribution:")
print(activity_df['Churn_Risk'].value_counts())

print(f"\n Monthly Active Users (sample):")
monthly_active = activity_df.groupby(
    'Month_Label'
)['Is_Active'].sum().reset_index()
monthly_active.columns = ['Month', 'Active_Students']
print(monthly_active.to_string(index=False))

print(f"\n Activity by Education Level:")
edu_activity = activity_df.groupby(
    'Education_Level'
).agg(
    Avg_Logins   = ('Login_Count',      'mean'),
    Avg_Q_Viewed = ('Questions_Viewed', 'mean'),
    Avg_Sessions = ('Expert_Sessions',  'mean'),
    Avg_Score    = ('Activity_Score',   'mean')
).round(1)
print(edu_activity)

print(f"\n First 5 rows:")
print(activity_df[[
    'Student_ID', 'Month_Label',
    'Login_Count', 'Questions_Viewed',
    'Expert_Sessions', 'Activity_Score',
    'Churn_Risk'
]].head().to_string(index=False))

# Save 
save_path = os.path.join(
    data_dir, 'student_activity.csv'
)
activity_df.to_csv(save_path, index=False)

print(f"\n student_activity.csv saved! "
      f"Rows: {len(activity_df):,}")

 Tracking months: 18
   From: 2023-01 To: 2024-06

 student_activity.csv generating...

 Total activity rows: 30,088
   Unique students:    7,627
   Months tracked:     18

 Activity Stats:
   Avg Logins/Month:   11.0
   Avg Q Viewed/Month: 15.0
   Avg Sessions/Month: 3.4
   Avg Activity Score: 20.5

 Churn Risk Distribution:
Churn_Risk
High      16992
Medium    12312
Low         784
Name: count, dtype: int64

 Monthly Active Users (sample):
  Month  Active_Students
2023-01             1465
2023-02             1519
2023-03             1617
2023-04             1674
2023-05             1758
2023-06             1696
2023-07             1716
2023-08             1740
2023-09             1728
2023-10             1715
2023-11             1702
2023-12             1715
2024-01             1701
2024-02             1618
2024-03             1614
2024-04             1631
2024-05             1660
2024-06             1454

 Activity by Education Level:
                 Avg_Logins  Avg_Q_Viewed  Avg_S

# EDA & Quality Check

In [15]:
# EDA & DATA QUALITY CHECK
# PURPOSE: Validate data before analysis
# In real projects this step finds issues
# before they affect results

print("=" * 55)
print("DATA QUALITY REPORT")
print("=" * 55)

# Missing values check
print("\n1. MISSING VALUES CHECK:")
for name, df in [
    ("students",     students_df),
    ("subscriptions",subs_df),
    ("mau_monthly",  mau_df),
    ("activity",     activity_df),
]:
    nulls = df.isnull().sum().sum()
    status = "Clean" if nulls == 0 \
             else f" {nulls} nulls"
    print(f"   {name:15s}: {status}")

# Duplicate check
print("\n2. DUPLICATE ROWS CHECK:")
for name, df, col in [
    ("students",      students_df,  "Student_ID"),
    ("subscriptions", subs_df,      "Sub_ID"),
]:
    dups = df.duplicated(subset=[col]).sum()
    status = "No duplicates" if dups == 0 \
             else f" {dups} duplicates"
    print(f"   {name:15s}: {status}")

# Data ranges check
print("\n3. DATE RANGE CHECK:")
print(f"   students reg:  "
      f"{students_df['Registration_Date'].min()} "
      f"to {students_df['Registration_Date'].max()}")
print(f"   subscriptions: "
      f"{subs_df['Sub_Start_Date'].min()} "
      f"to {subs_df['Sub_Start_Date'].max()}")
print(f"   mau_monthly:   "
      f"{mau_df['Month_Label'].min()} "
      f"to {mau_df['Month_Label'].max()}")

# Business metrics check
print("\n4. KEY BUSINESS METRICS:")
print(f"   Total Students:       "
      f"{len(students_df):,}")
print(f"   Total Subscriptions:  "
      f"{len(subs_df):,}")
print(f"   Overall Churn Rate:   "
      f"{(subs_df['Churned']=='Yes').mean()*100:.1f}%")
print(f"   Total Revenue:        "
      f"${subs_df['Revenue_USD'].sum():,.2f}")
print(f"   Peak MAU:             "
      f"{mau_df['MAU'].max():,}")
print(f"   Lowest MAU:           "
      f"{mau_df['MAU'].min():,}")
drop = (1-mau_df['MAU'].min()/
        mau_df['MAU'].max())*100
print(f"   MAU Decline:          {drop:.1f}%")

# Churn reason distribution
print("\n5. CHURN REASON DISTRIBUTION:")
churn_dist = subs_df[
    subs_df['Churned']=='Yes'
]['Churn_Reason'].value_counts()
total_churn = churn_dist.sum()
for reason, count in churn_dist.items():
    pct = count/total_churn*100
    print(f"   {reason:30s}: "
          f"{count:,} ({pct:.1f}%)")

# Cohort analysis
print("\n6. COHORT CHURN ANALYSIS:")
def get_cohort(month):
    if month < '2023-05':
        return 'Pre-AI (Before May 23)'
    elif month < '2023-11':
        return 'ChatGPT Impact (May-Oct 23)'
    elif month < '2024-01':
        return 'Steep Decline (Nov-Dec 23)'
    else:
        return 'Crisis Phase (2024)'

subs_df['Cohort'] = subs_df[
    'Sub_Start_Month'
].apply(get_cohort)

cohort_stats = subs_df.groupby('Cohort').agg(
    Total   = ('Sub_ID',  'count'),
    Churned = ('Churned', lambda x:
               (x=='Yes').sum())
).reset_index()
cohort_stats['Churn_Rate'] = (
    cohort_stats['Churned'] /
    cohort_stats['Total'] * 100
).round(1)
print(cohort_stats[[
    'Cohort', 'Total', 'Churn_Rate'
]].to_string(index=False))

# Plan analysis
print("\n7. PLAN TYPE ANALYSIS:")
plan_stats = subs_df.groupby('Plan_Type').agg(
    Total     = ('Sub_ID',       'count'),
    Churned   = ('Churned',
                 lambda x: (x=='Yes').sum()),
    Revenue   = ('Revenue_USD',  'sum'),
    Avg_Days  = ('Days_Active',  'mean')
).reset_index()
plan_stats['Retention'] = (
    (1 - plan_stats['Churned'] /
     plan_stats['Total']) * 100
).round(1)
print(plan_stats[[
    'Plan_Type', 'Total',
    'Retention', 'Revenue'
]].to_string(index=False))

# Activity risk analysis
print("\n8. ACTIVITY RISK DISTRIBUTION:")
risk_dist = activity_df[
    'Churn_Risk'
].value_counts()
for risk, count in risk_dist.items():
    pct = count/len(activity_df)*100
    print(f"   {risk:10s}: "
          f"{count:,} ({pct:.1f}%)")

print("\n" + "="*55)
print(" Data Quality Check Complete!")
print("="*55)

DATA QUALITY REPORT

1. MISSING VALUES CHECK:
   students       : Clean
   subscriptions  :  15454 nulls
   mau_monthly    : Clean
   activity       : Clean

2. DUPLICATE ROWS CHECK:
   students       : No duplicates
   subscriptions  : No duplicates

3. DATE RANGE CHECK:
   students reg:  2022-06-01 to 2024-06-30
   subscriptions: 2022-06-03 to 2024-06-30
   mau_monthly:   2023-01 to 2024-06

4. KEY BUSINESS METRICS:
   Total Students:       10,000
   Total Subscriptions:  12,626
   Overall Churn Rate:   38.8%
   Total Revenue:        $210,929.21
   Peak MAU:             4,541,086
   Lowest MAU:           1,350,000
   MAU Decline:          70.3%

5. CHURN REASON DISTRIBUTION:
   Found Better Alternative      : 1,910 (39.0%)
   Price Too High                : 1,195 (24.4%)
   Not Useful Anymore            : 785 (16.0%)
   Exam Season Over              : 617 (12.6%)
   Other                         : 206 (4.2%)
   Technical Issues              : 186 (3.8%)

6. COHORT CHURN ANALYSIS:
   

# Summery Report

In [16]:
# SUMMARY: Final project data summary
# PURPOSE: Document what was generated

print("=" * 55)
print("PROJECT DATA SUMMARY")
print("EdTech Subscription & Churn Analysis")
print("=" * 55)

print(f"""
FILES GENERATED:
   students.csv          → {len(students_df):,} rows
   subscriptions.csv     → {len(subs_df):,} rows
   mau_monthly.csv       → {len(mau_df)} rows
   student_activity.csv  → {len(activity_df):,} rows

LOCATION: {data_dir}

KEY FINDINGS FROM EDA:
   Total Students:     {len(students_df):,}
   Total Subs:         {len(subs_df):,}
   Overall Churn:      {(subs_df['Churned']=='Yes').mean()*100:.1f}%
   Total Revenue:      ${subs_df['Revenue_USD'].sum():,.0f}
   MAU Peak:           {mau_df['MAU'].max():,}
   MAU Lowest:         {mau_df['MAU'].min():,}
   MAU Decline:        {(1-mau_df['MAU'].min()/mau_df['MAU'].max())*100:.1f}%

TOP CHURN REASON:
   {subs_df[subs_df['Churned']=='Yes']['Churn_Reason'].value_counts().index[0]}

BEST RETENTION PLAN:
   {plan_stats.loc[plan_stats['Retention'].idxmax(), 'Plan_Type']} — {plan_stats['Retention'].max()}%

NEXT STEPS:
   1. Import CSVs to MySQL → Run SQL queries
   2. Open Power BI → Load CSVs
   3. Build interactive dashboard
   4. Upload to GitHub
""")

print("Project complete — ready for SQL & Power BI!")

PROJECT DATA SUMMARY
EdTech Subscription & Churn Analysis

FILES GENERATED:
   students.csv          → 10,000 rows
   subscriptions.csv     → 12,626 rows
   mau_monthly.csv       → 18 rows
   student_activity.csv  → 30,088 rows

LOCATION: G:\01_Learning\Projects\01_My Projects\08_Chegg Inc\data

KEY FINDINGS FROM EDA:
   Total Students:     10,000
   Total Subs:         12,626
   Overall Churn:      38.8%
   Total Revenue:      $210,929
   MAU Peak:           4,541,086
   MAU Lowest:         1,350,000
   MAU Decline:        70.3%

TOP CHURN REASON:
   Found Better Alternative

BEST RETENTION PLAN:
   Annual — 72.9%

NEXT STEPS:
   1. Import CSVs to MySQL → Run SQL queries
   2. Open Power BI → Load CSVs
   3. Build interactive dashboard
   4. Upload to GitHub

Project complete — ready for SQL & Power BI!
